In [1]:
print("Runnin the notebook")

Runnin the notebook


In [5]:
import os
import random
import requests
from typing import Dict, Any, List, TypedDict, Annotated

from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool

from langchain_groq import ChatGroq

from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

True

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    groq_api_key= os.getenv("GROQ_API_KEY")
)

In [ ]:
search_tool = DuckDuckGoSearchRun(region="us-en")
alpha_key = os.getenv("ALPHA_VANTAGE_KEY")

@tool
def get_stock_price(symbol: str) -> dict:
    """
        Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
        using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={alpha_key}"
    r = requests.get(url)
    return r.json()
# get_stock_price.invoke({"symbol": "TSLA"})

{'Global Quote': {'01. symbol': 'TSLA',
  '02. open': '357.0700',
  '03. high': '358.8000',
  '04. low': '345.2000',
  '05. price': '348.7500',
  '06. volume': '32864721',
  '07. latest trading day': '2026-08-28',
  '08. previous close': '354.8100',
  '09. change': '-6.0600',
  '10. change percent': '-1.7080%'}}

AIMessage(content='1. Boil fresh water and let it cool slightly to about 195‑205\u202f°F (90‑96\u202f°C).  \n2. Measure 1\u202fto\u202f2\u202ftablespoons of coarsely ground coffee per 6\u202foz of water and place it in a filter.  \n3. Slowly pour the hot water over the grounds, ensuring even saturation, and let it steep for 3‑4\u202fminutes.  \n4. Remove the filter, discard the grounds, and pour the brewed black coffee into a pre‑warmed mug.  \n5. Enjoy the black coffee as is, or add optional sweetener or milk to taste.', additional_kwargs={'reasoning_content': 'We need to follow developer instruction: break the request into 3-5 short, actionable tasks, each task one line only. The user asked "how to make black coffee". So we should provide steps broken into tasks, each line one line. Probably 4 tasks: boil water, grind coffee, add coffee to filter, pour water, let brew, serve. Provide 4-5 lines.\n\nMake sure each line is a task, concise. No extra commentary.'}, response_metadata={'tok

str

In [6]:
tasks = []

for t in response.content.split("\n"):
    t = t.strip("-• ").strip()

    if len(t) > 5:
        tasks.append(t)

    if len(tasks) == 5:
        break

print(tasks)

['1. Boil fresh water and let it cool slightly to about 195‑205\u202f°F (90‑96\u202f°C).', '2. Measure 1\u202fto\u202f2\u202ftablespoons of coarsely ground coffee per 6\u202foz of water and place it in a filter.', '3. Slowly pour the hot water over the grounds, ensuring even saturation, and let it steep for 3‑4\u202fminutes.', '4. Remove the filter, discard the grounds, and pour the brewed black coffee into a pre‑warmed mug.', '5. Enjoy the black coffee as is, or add optional sweetener or milk to taste.']


In [7]:

prompt = ChatPromptTemplate.from_messages([
            ("system", "Break the request into 3-5 short, actionable tasks."
            "Each task must be one line only."),
            ("user", user_query)
        ])
chain = prompt | llm
response = chain.invoke({})
response

AIMessage(content='1. Boil fresh water to just below the boiling point (around 195‑205°F / 90‑96°C).  \n2. Grind coffee beans to a medium‑fine consistency (about the texture of sand).  \n3. Place the ground coffee in a filter or French press (≈1\u202fto\u202f2\u202ftablespoons per 6\u202foz water).  \n4. Pour the hot water over the grounds, let steep for 3‑4\u202fminutes, then serve the black coffee.', additional_kwargs={'reasoning_content': 'The user asks "how to make black coffee". The developer instruction: "Break the request into 3-5 short, actionable tasks. Each task must be one line only."\n\nThus we need to respond with a list of 3-5 short, actionable tasks, each on its own line. Provide steps to make black coffee. Probably 4 tasks: boil water, grind beans, brew, pour. Each line.\n\nWe must obey developer instruction. No extra text. Probably just bullet points or plain lines. Ensure each line is one line, no extra sentences.\n\nLet\'s produce 4 tasks.'}, response_metadata={'toke

In [11]:

prmp = ChatPromptTemplate.from_messages(
   
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain RAG to me.line by line"
        }
    ]
)

In [13]:
chain = prmp | llm
response = chain.invoke({})
response.content

'**Retrieval‑Augmented Generation (RAG) – a line‑by‑line walk‑through**\n\n| # | Concept / Step | What it means (in plain language) |\n|---|----------------|-----------------------------------|\n| 1 | **Goal** | Build a language model that can answer questions **with up‑to‑date, factual information** instead of only relying on what it memorized during training. |\n| 2 | **Two main pieces** | 1️⃣ **Retriever** – a search engine that finds relevant documents.<br>2️⃣ **Generator** – a generative LLM that reads those documents and writes the final answer. |\n| 3 | **Knowledge base (corpus)** | A large collection of texts (e.g., Wikipedia articles, product manuals, internal docs). It can be static or refreshed continuously. |\n| 4 | **Indexing the corpus** | The retriever first builds an index (e.g., FAISS, Elasticsearch) that maps each passage to a dense vector representation, enabling fast similarity search. |\n| 5 | **User query arrives** | The system receives a question like “What are t

In [15]:
response.usage_metadata

{'input_tokens': 88,
 'output_tokens': 1142,
 'total_tokens': 1230,
 'output_token_details': {'reasoning': 51}}